# Load and export the shoe-classification dataset

Download `keremberke/shoe-classification` from Hugging Face, inspect its labels and splits, then export it to the folder format expected by YOLO11 classification.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from datasets import load_dataset

DATASET_ID = "keremberke/shoe-classification"
DATASET_CONFIG = "full"
EXPECTED_CLASSES = {"adidas", "converse", "nike"}
EXPECTED_SPLIT_SIZES = {"train": 576, "validation": 166, "test": 83}
SPLIT_DIRECTORY_NAMES = {"train": "train", "validation": "val", "test": "test"}
EXPORT_ROOT = Path("../datasets/shoe_brands").resolve()

In [ ]:
ds = load_dataset(DATASET_ID, DATASET_CONFIG, trust_remote_code=True)
ds

In [ ]:
actual_split_sizes = {split_name: len(split) for split_name, split in ds.items()}
if actual_split_sizes != EXPECTED_SPLIT_SIZES:
    raise ValueError(
        f"Unexpected dataset split sizes: {actual_split_sizes}; "
        f"expected {EXPECTED_SPLIT_SIZES}."
    )

label_feature = ds["train"].features["labels"]
label_names = label_feature.names
if set(label_names) != EXPECTED_CLASSES:
    raise ValueError(f"Unexpected classes: {label_names}")

print(f"Features: {ds['train'].features}")
print(f"Labels: {label_names}")
print(f"Split sizes: {actual_split_sizes}")
print(f"Total images: {sum(actual_split_sizes.values())}")

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, example in zip(axes, ds["train"].select(range(3))):
    axis.imshow(example["image"])
    axis.set_title(label_names[example["labels"]])
    axis.axis("off")
plt.tight_layout()

In [ ]:
def export_split(split_name: str) -> Counter:
    output_split_name = SPLIT_DIRECTORY_NAMES[split_name]
    exported_counts = Counter()

    for index, example in enumerate(ds[split_name]):
        class_name = label_names[example["labels"]]
        class_directory = EXPORT_ROOT / output_split_name / class_name
        class_directory.mkdir(parents=True, exist_ok=True)

        output_path = class_directory / f"{split_name}_{index:04d}.jpg"
        example["image"].convert("RGB").save(output_path, format="JPEG", quality=95)
        exported_counts[class_name] += 1

    return exported_counts


export_summary = {
    SPLIT_DIRECTORY_NAMES[split_name]: export_split(split_name)
    for split_name in SPLIT_DIRECTORY_NAMES
}
export_summary

In [ ]:
exported_total = sum(sum(class_counts.values()) for class_counts in export_summary.values())
expected_total = sum(EXPECTED_SPLIT_SIZES.values())
if exported_total != expected_total:
    raise RuntimeError(f"Exported {exported_total} images; expected {expected_total}.")

missing_classes_by_split = {
    split_name: sorted(EXPECTED_CLASSES - set(class_counts))
    for split_name, class_counts in export_summary.items()
    if set(class_counts) != EXPECTED_CLASSES
}

print(f"Dataset exported to: {EXPORT_ROOT}")
for split_name, class_counts in export_summary.items():
    print(f"{split_name}: {dict(sorted(class_counts.items()))}")
print(f"Total exported images: {exported_total}")
if missing_classes_by_split:
    print(f"WARNING: splits with missing classes: {missing_classes_by_split}")

Dataset is ready for `../yolo11/setup_yolo11.ipynb`. No training is performed in this notebook.